| Параметр | Что оценивает | Пример |
|----------|---------------|--------|
| **Sentiment** (тональность) | Эмоциональную **валентность** (positive / neutral / negative) | «приятный запах» → positive, «вонь» → negative |
| **Connotation** (коннотация) | **Культурно-стилистический ореол** (высокое / низкое / нейтральное) | «аромат» → высокая коннотация, «вонь» → низкая, «запах» → нейтральная |



## Sentiment

### # Установка (если нужно)
### !pip install transformers torch pandas openpyxl


In [27]:
FILENAME = '../results/Разметка.xlsx'

In [28]:
import pandas as pd
from openpyxl import load_workbook

from openpyxl.utils import get_column_letter

import torch
from transformers import pipeline
from sklearn.metrics import classification_report, cohen_kappa_score


In [29]:
df = pd.read_excel(FILENAME, sheet_name='Качество', header=0)

# Посмотрим на данные
print(f"Загружено {len(df)} предложений")
print(df[['sent_text_RU', 'sent_text_EN']].head(3))



Загружено 162 предложений
                                        sent_text_RU  \
0                    В душных комнатах пахло мятой .   
1  Красные его цветы и листья с колючками издавал...   
2               Брусок издавал тончайший запах роз .   

                                        sent_text_EN  
0                 The stuffy rooms smelled of mint .  
1  In the heat their red flowers and prickly leav...  
2     The bar gave off the faintest smell of roses .  


### Загрузка моделей

In [30]:
# Русская модель (3 класса: POSITIVE, NEUTRAL, NEGATIVE)
print("\nЗагрузка русской модели...")
ru_pipeline = pipeline(
    "sentiment-analysis",
    model="blanchefort/rubert-base-cased-sentiment",
    tokenizer="blanchefort/rubert-base-cased-sentiment",
    device=0 if torch.cuda.is_available() else -1  # GPU если есть
)

# # Английская модель (2 класса: POSITIVE, NEGATIVE)
# print("Загрузка английской модели...")
# en_pipeline = pipeline(
#     "sentiment-analysis",
#     model="distilbert-base-uncased-finetuned-sst-2-english",
#     tokenizer="distilbert-base-uncased-finetuned-sst-2-english",
#     device=0 if torch.cuda.is_available() else -1
# )


Загрузка русской модели...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6263.60it/s]


In [31]:
# Просто заменить английскую модель на ту, что поддерживает 3 класса
print("Загрузка английской модели с 3 классами...")
en_pipeline = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=0 if torch.cuda.is_available() else -1
)

Загрузка английской модели с 3 классами...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 24963.14it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Для русских текстов
print("\nАнализ русских текстов...")
ru_results = []
for text in df['sent_text_RU'].tolist():
    try:
        result = ru_pipeline(text[:512])[0]  # обрезаем до 512 токенов
        ru_results.append({
            'label': result['label'].lower(),
            'confidence': result['score']
        })
    except:
        ru_results.append({'label': 'NEUTRAL', 'confidence': 0.0})



Анализ русских текстов...


In [ ]:
# Для английских текстов
print("Анализ английских текстов...")
en_results = []
for text in df['sent_text_EN'].tolist():
    try:
        result = en_pipeline(text[:512])[0]
        en_results.append({
            'label': result['label'].lower(),
            'confidence': result['score']
        })
    except:
        en_results.append({'label': 'NEGATIVE', 'confidence': 0.0})



Анализ английских текстов...


In [ ]:
df['sentiment_RU'] = [r['label'] for r in ru_results]
df['confidence_RU'] = [r['confidence'] for r in ru_results]
df['sentiment_EN'] = [r['label'] for r in en_results]
df['confidence_EN'] = [r['confidence'] for r in en_results]

# df 

### Сохранение

In [41]:
# ПРОСТОЙ СПОСОБ: перезаписать лист целиком

with pd.ExcelWriter(FILENAME, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    # Выбираем нужные колонки для сохранения
    columns_to_save = [
        'sent_text_RU',
        'confidence_RU', 
        'sentiment_RU', 
        'sentiment_EN',
        'confidence_EN',
        'sent_text_EN',
    ]
    
    # Сохраняем только выбранные колонки
    df[columns_to_save].to_excel(
        writer, 
        sheet_name='Авторазметка', 
        index=False
    )
    
print(f"✅ Лист полностью перезаписан с {len(df)} строками")

✅ Лист полностью перезаписан с 162 строками


In [36]:
# Если хотим сохранить вообще отдельным файлом
# df.to_excel('Test', index=False)